# Agentic RAG Multi-Agent Demonstration

This notebook demonstrates a Router Agent and Retriever Agent coordinating across a static PDF source, fresh web search, and an optional direct-generation path. The first section is an executed, key-free verification. The final section shows the live CrewAI workflow.


## Architecture and agent responsibilities

- **Router Agent:** analyzes the question, chooses `pdf`, `web`, or `generate`, and records its rationale.
- **Retriever Agent:** invokes the selected tool and returns a grounded answer with sources.
- **Trace logger:** writes structured JSONL events for routing, retrieval, and answer production.

CrewAI is used because its role/task model makes the handoff explicit and keeps tool access scoped to the agent that needs it.


In [1]:
from pathlib import Path
from src.agentic_rag import AgenticRAGSystem, offline_answer, route_question

TRACE = Path("logs/notebook_trace.jsonl")
if TRACE.exists():
    TRACE.unlink()
system = AgenticRAGSystem(Path("data/attention_is_all_you_need.pdf"), TRACE)


## Demo 1 — PDF route


In [2]:
question = "How does multi-head attention work in the paper?"
result = offline_answer(question)
system.route(question)
print("Route:", result["route"])
print("Rationale:", result["rationale"])
print("Answer:", result["answer"])
print("Source:", result["sources"][0])


Route: pdf
Rationale: The question targets concepts in the domain PDF.
Answer: The Transformer replaces recurrence with attention. Multi-head attention projects queries, keys, and values into several learned subspaces, performs scaled dot-product attention in parallel, then concatenates the results.
Source: Vaswani et al., Attention Is All You Need, Sections 3.2–3.3


## Demo 2 — Web route

The offline run records the intended behavior without fabricating current results. With `TAVILY_API_KEY` configured, the live Crew delegates this question to `TavilySearchResults`.


In [3]:
question = "What are the latest Transformer efficiency results?"
result = offline_answer(question)
system.route(question)
print("Route:", result["route"])
print("Rationale:", result["rationale"])
print("Answer:", result["answer"])
print("Source:", result["sources"][0])


Route: web
Rationale: The question asks for time-sensitive information.
Answer: This route requires a live search because the answer can change. In offline mode the system records the intended Tavily query rather than inventing results.
Source: Tavily live-search placeholder


## Trace inspection


In [4]:
import json
for line in TRACE.read_text(encoding="utf-8").splitlines():
    event = json.loads(line)
    print(event["event"], "|", event["route"], "|", event["question"])


router_decision | pdf | How does multi-head attention work in the paper?
router_decision | web | What are the latest Transformer efficiency results?


## Live CrewAI run

Create `.env` from `.env.example`, add the required keys, place the PDF in `data/`, then enable and run the cell below. The Retriever Agent receives `PDFSearchTool` for document questions and `TavilySearchResults` for time-sensitive questions.


In [ ]:
# from dotenv import load_dotenv
# load_dotenv()
# live_question = "What does the paper say about positional encoding?"
# crew = system.build_crew(live_question)
# print(crew.kickoff())


## Verification result

The executed examples demonstrate both required retrieval routes and expose the router's contribution. Live retrieval remains opt-in because API keys are never embedded in notebooks or trace files.
